# 2.3 µm Iterative MFによる背景領域選択と人工プルーム二吸収帯評価

実メタンプルームを含むHISUIシーンに対して、次の順序で評価

1. **元シーンの2.3 µm帯にIterative Matched Filterを適用**し、実プルーム候補を背景統計から除外する
2. 2.3 µm帯のMF応答が背景中心に近く、空間的にも安定した場所を人工プルーム注入位置として自動選択する
3. 選んだ場所へ、MODTRAN絶対濃度LUTから作った相対放射輝度比を用いて人工プルームを注入する
4. 注入後のシーンに対して、1.6 µm帯と2.3 µm帯を**独立にIterative MF**で評価する
5. 2.3 µm帯の候補を1.6 µm帯が同一画素または近傍で支持するかを調べる
6. 既知の人工プルーム真値に対して、単一帯域と二吸収帯相互確認の性能を比較する

ここで「低い2.3 µm MF応答」は、最小の負値ではなく、**robust Z-scoreが0付近でメタンらしい正応答がない領域**と解釈。強い負の外れ値は、影・地表異常・校正誤差の可能性があるため注入場所には使わない。

主な評価対象は次の4つ。

- 1.6 µm帯MF単独
- 2.3 µm帯MF単独
- 二帯域の同一画素AND
- 2.3 µm候補を1.6 µm帯が近傍で支持する相互確認型MF

元シーンに実プルームがあるため、人工注入の評価は、元シーンで既に検出されていた候補を差し引いた**新規検出マスク**と、人工プルーム周辺の局所評価領域を用いて行う。

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, label, maximum_filter
from scipy.stats import rankdata
np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
# 入力
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"
OUTPUT_DIR = Path("./dual_window_background_injection_output")

# MODTRAN・HISUI
BACKGROUND_CH4_PPM = 1.8
FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5
UAS_STEP_PPM = 0.05

# Iterative MF 
MAX_ITERATIONS_BASELINE = 12
MAX_ITERATIONS_INJECTION = 10
EXCLUSION_Z_16 = 2.5
EXCLUSION_Z_23 = 2.5
EXCLUSION_DILATION_PIXELS = 2
MIN_BACKGROUND_FRACTION = 0.55
MIN_BACKGROUND_PIXELS = 300
CONVERGENCE_NEW_PIXEL_FRACTION = 2.5e-4
COVARIANCE_SHRINKAGE = 0.08
COVARIANCE_RIDGE_RELATIVE = 1e-8

# 人工プルーム注入場所の選択 
# Noneなら2.3 µm Iterative MFから自動選択。指定する場合はROI配列内の(row, col)
INJECTION_CENTER_YX = None
CENTER_SEARCH_STRIDE = 2
MAX_ABS_BASELINE_Z23 = 0.75
MAX_LOCAL_MAX_Z23 = 1.5
REAL_PLUME_EXCLUSION_BUFFER_PIXELS = 4

# 1.6 µm帯の既存異常が注入評価を汚さないための安全確認
# 主選択スコアは2.3 µm帯だけで作る
USE_16_AS_SAFETY_CHECK = False
MAX_ABS_BASELINE_Z16 = 1.5
MAX_LOCAL_ABS_Z16 = 2.0

# 人工プルーム形状 
PLUME_ANGLE_DEG = 0.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0
PLUME_SUPPORT_FRACTION_FOR_LOCATION = 0.05

# LUT範囲外のピークは自動的に除外
INJECTION_PEAKS_PPM = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
PRIMARY_PEAK_PPM = 2.0

# 最終候補抽出 
DETECTION_Z_16 = 2.0
DETECTION_Z_23 = 3.0
NEIGHBORHOOD_RADIUS = 1
MIN_REGION_PIXELS = 3
BASELINE_CANDIDATE_BUFFER_PIXELS = 2

# 人工真値・局所評価
TRUE_MASK_FRACTION_OF_PEAK = 0.10
TRUE_MASK_MIN_ENHANCEMENT_PPM = 0.05
EVALUATION_BUFFER_PIXELS = 12
TOP_LOCATION_CANDIDATES = 30

# 主ピークでしきい値探索
THRESHOLDS_16 = np.arange(0.5, 4.01, 0.5)
THRESHOLDS_23 = np.arange(1.5, 5.01, 0.5)

RANDOM_SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. HISUI ROIスペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError("wave_***nm形式の列が見つかりません。")
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {value: i for i, value in enumerate(ys)}
    x_to_i = {value: i for i, value in enumerate(xs)}

    cube = np.full((len(ys), len(xs), spectra.shape[1]), fill_value, dtype=float)
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(cube, nodata_values=(0.0, -9999.0), require_positive=True):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)


df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_original, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_original)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_original.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)
